# 02 — Observation sets
* **D_prod**: every scenario × 50 windows under production conditions (inlet ±2 K measured with 0.1 K
  offsets, Ci 1%, Q 0.5%, every sensor calibration offset drawn), raw windows stored.
* **D_nom**: the same under nominal conditions (nominal inlet, no calibration offsets).
* **L_prod**: 1000 windows per closed-loop scenario under production conditions (headline accuracies).
Seeds fixed here; no posterior involved. ~5 min.

In [ ]:
import json, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "notebooks" and NB_DIR.parent.name == "system_III", f"run from system_III/notebooks (cwd {NB_DIR})"
ROOT = NB_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, system_iii
from system_iii import physics as ph, simulator as sim, features as fe, data as da, scenarios as scn
from system_iii import sensors as sen, information as info, config as cfg, rules, metrics as wm
for _m in (ph, sim, fe, da, scn, sen, info, cfg, rules, wm):
    importlib.reload(_m)
R, FIGS, DATA = ROOT / "results", ROOT / "figures", ROOT / "data"
for _d in (R, FIGS, DATA):
    _d.mkdir(exist_ok=True)
P = ph.PLANT
T0 = time.time()

In [ ]:
names = scn.CLOSED + scn.OPEN
dprod = da.scenario_set(names, 50, seed=2000, conditions="production", return_raw=True)
dnom = da.scenario_set(names, 50, seed=2000, conditions="nominal", return_raw=True)
da.save_npz(DATA / "D_prod.npz", dprod); da.save_npz(DATA / "D_nom.npz", dnom)
lprod = da.scenario_set(scn.CLOSED, 1000, seed=2500, conditions="production", return_raw=False)
da.save_npz(DATA / "L_prod.npz", lprod)
print({k: v.shape for k, v in dprod.items()}, lprod["x"].shape, f"{(time.time() - T0) / 60:.1f} min")

In [ ]:
ia, ib = fe.NAMES["S1m"].index("ln_alpha_hat"), fe.NAMES["S1m"].index("ln_beta_hat_jac")
tab = pd.DataFrame({k: {"alpha_hat": np.exp(lprod["x"][lprod["sc_name"] == k, ia]).mean(),
                        "beta_hat_jac": np.exp(lprod["x"][lprod["sc_name"] == k, ib]).mean()} for k in scn.CLOSED}).T
display(tab.round(4)); tab.to_csv(R / "02_proxy_means_L.csv")

## Interpretation — to be written after the run
Write toward the storyline (`docs/system_III_plan.md` §1) and update the claims ledger (§2).